<a href="https://colab.research.google.com/github/rachmadnanda/MachineLearning2026/blob/main/jobsheet_pbl/JS03_PBL_244107020095_RACHMAD_FEBRIANANDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# JS03 — Seleksi Fitur dan Encoding pada Dataset RumahPoint

Jobsheet asli membahas pemilihan kolom, encoding, dan standardisasi pada data Titanic.
Versi PBL ini menguji fitur mana yang benar-benar membantu memprediksi **harga** dan
**kelas harga** properti, sekaligus membandingkan tiga cara encoding `lokasi`.

## Langkah 0 — Muat dan Bersihkan Data

Fungsi pembersihan dibuat identik dengan JS02 supaya hasil antar-notebook konsisten.

In [1]:
import os, warnings
warnings.filterwarnings('ignore')
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '3')
os.environ.setdefault('TF_ENABLE_ONEDNN_OPTS', '0')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.feature_selection import mutual_info_regression, mutual_info_classif, f_regression
from sklearn.metrics import mean_squared_error, accuracy_score

pd.set_option('display.width', 120)
sns.set_theme(style='whitegrid', palette='muted')
RANDOM_STATE = 42
TAHUN_INI = 2026


def muat_data_properti(path='dataset/properti_multi_kota.csv'):
    df = pd.read_csv(path).drop_duplicates()
    df = df[(df['luas_m2'] >= 10) & (df['luas_m2'] <= 2000)]
    df = df[df['luas_bangunan'].isna() |
            ((df['luas_bangunan'] >= 10) & (df['luas_bangunan'] <= 2000))]
    df = df[df['harga'].between(50_000_000, 100_000_000_000)]
    df = df[df['jumlah_kamar'].isna() | (df['jumlah_kamar'] <= 20)]
    df = df[df['jumlah_kamar_mandi'].isna() | (df['jumlah_kamar_mandi'] <= 20)]

    tahun = df['usia_bangunan'] > 100
    df.loc[tahun, 'usia_bangunan'] = TAHUN_INI - df.loc[tahun, 'usia_bangunan']
    df.loc[~df['usia_bangunan'].between(0, 100) & df['usia_bangunan'].notna(),
           'usia_bangunan'] = np.nan

    df['kondisi'] = np.where(df['usia_bangunan'].isna(), 'Tidak diketahui',
                     np.where(df['usia_bangunan'] <= 3, 'Baru', 'Bekas'))

    q1, q2 = df['harga'].quantile([1 / 3, 2 / 3])
    df['kelas_harga'] = np.where(df['harga'] < q1, 'Murah',
                         np.where(df['harga'] <= q2, 'Sedang', 'Mahal'))

    fitur = ['luas_m2', 'luas_bangunan', 'jumlah_kamar', 'jumlah_kamar_mandi']
    df[fitur] = df[fitur].fillna(df[fitur].median())

    df['lokasi_bersih'] = df['lokasi'].str.strip().str.title()
    frek = df['lokasi_bersih'].value_counts(normalize=True)
    df['lokasi_freq'] = df['lokasi_bersih'].map(frek)
    return df


df = muat_data_properti()
print(f'Baris siap analisis: {len(df):,}')
df[['luas_m2', 'luas_bangunan', 'jumlah_kamar', 'jumlah_kamar_mandi',
    'kondisi', 'lokasi_freq', 'harga', 'kelas_harga']].head()

Baris siap analisis: 56,170


,luas_m2,luas_bangunan,jumlah_kamar,jumlah_kamar_mandi,kondisi,lokasi_freq,harga,kelas_harga
0,30.0,65.0,3.0,3.0,Bekas,0.022610,2.037543e+08,Murah
1,122.0,65.0,5.0,4.0,Bekas,0.032989,1.058032e+09,Sedang
2,142.8,65.0,1.0,1.0,Bekas,0.032989,1.335793e+09,Mahal
3,152.6,65.0,4.0,3.0,Bekas,0.032989,1.613771e+09,Mahal
4,61.2,65.0,1.0,1.0,Bekas,0.025636,5.221562e+08,Sedang


## Langkah 1 — Korelasi Fitur Numerik dengan Harga

Korelasi Pearson dipakai untuk melihat hubungan **linear**. Karena harga sangat miring,
korelasi juga dihitung terhadap `log10(harga)`.

In [2]:
# usia_bangunan tidak dipakai sebagai fitur: >92% kosong (lihat JS02).
# Informasinya sudah diringkas menjadi kolom kategorikal `kondisi`.
FITUR_NUM = ['luas_m2', 'luas_bangunan', 'jumlah_kamar', 'jumlah_kamar_mandi']
df['log_harga'] = np.log10(df['harga'])

korelasi = df[FITUR_NUM + ['harga', 'log_harga']].corr(numeric_only=True)
print(korelasi[['harga', 'log_harga']].drop(['harga', 'log_harga']).round(4).to_string())

plt.figure(figsize=(7.5, 5.5))
sns.heatmap(korelasi, annot=True, fmt='.3f', cmap='coolwarm', center=0,
            square=True, cbar_kws={'shrink': 0.8})
plt.title('Matriks korelasi fitur numerik')
plt.tight_layout()
plt.show()

                     harga  log_harga
luas_m2             0.5144     0.4932
luas_bangunan       0.6370     0.6359
jumlah_kamar        0.3346     0.4902
jumlah_kamar_mandi  0.3979     0.6138


## Langkah 2 — Multikolinearitas (VIF)

VIF > 10 menandakan sebuah fitur hampir bisa dijelaskan linear oleh fitur lain. Nilai
dihitung dari invers matriks korelasi, tanpa memerlukan library tambahan.

In [3]:
def hitung_vif(frame):
    corr = frame.corr().values
    return pd.Series(np.diag(np.linalg.inv(corr)), index=frame.columns, name='VIF')

vif = hitung_vif(df[FITUR_NUM].dropna())
print(vif.round(2).to_string())
print('\nAturan praktis: VIF > 10 -> multikolinearitas tinggi')

luas_m2               1.92
luas_bangunan         2.37
jumlah_kamar          2.45
jumlah_kamar_mandi    2.61

Aturan praktis: VIF > 10 -> multikolinearitas tinggi


## Langkah 3 — Informasi Timbal Balik (Mutual Information)

MI mengukur ketergantungan **non-linear** antara fitur dan target. Berbeda dengan
korelasi Pearson, MI tetap bisa menangkap hubungan yang tidak lurus.

In [4]:
sampel = df.sample(15_000, random_state=RANDOM_STATE)

mi_reg = mutual_info_regression(sampel[FITUR_NUM], sampel['log_harga'],
                                random_state=RANDOM_STATE)
mi_kls = mutual_info_classif(sampel[FITUR_NUM], sampel['kelas_harga'],
                             random_state=RANDOM_STATE)

tabel_mi = pd.DataFrame({'MI vs log(harga)': mi_reg, 'MI vs kelas': mi_kls},
                        index=FITUR_NUM).sort_values('MI vs log(harga)', ascending=False)
print(tabel_mi.round(4).to_string())

                    MI vs log(harga)  MI vs kelas
luas_bangunan                 0.6025       0.3798
jumlah_kamar_mandi            0.3678       0.2961
jumlah_kamar                  0.3319       0.2437
luas_m2                       0.2536       0.1572


In [5]:
f_stat, p_val = f_regression(sampel[FITUR_NUM], sampel['log_harga'])
pd.DataFrame({'F': f_stat, 'p-value': p_val}, index=FITUR_NUM).round(6)

,F,p-value
luas_m2,4639.682389,0.0
luas_bangunan,10027.302403,0.0
jumlah_kamar,4548.223799,0.0
jumlah_kamar_mandi,8620.602416,0.0


## Langkah 4 — Perbandingan Set Fitur untuk Regresi

Empat kandidat set fitur diuji dengan **Ridge Regression** dan validasi silang 5-fold.
Targetnya `log10(harga)`; RMSE dilaporkan dalam satuan log agar bisa dibandingkan
dengan model lain di JS07/JS13.

In [6]:
from sklearn.model_selection import cross_val_predict

def evaluasi_regresi(kolom, label):
    X = df[kolom]
    y = df['log_harga']
    model = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
    pred = cross_val_predict(model, X, y, cv=KFold(5, shuffle=True, random_state=RANDOM_STATE))
    rmse = np.sqrt(mean_squared_error(y, pred))
    r2 = 1 - ((y - pred) ** 2).sum() / ((y - y.mean()) ** 2).sum()
    return {'Set fitur': label, 'Jumlah fitur': len(kolom), 'RMSE log': rmse, 'R2': r2}

kandidat = [
    (['luas_m2'], 'luas_m2 saja'),
    (['luas_m2', 'luas_bangunan'], 'luas tanah + bangunan'),
    (FITUR_NUM, 'semua numerik'),
    (['luas_m2', 'luas_bangunan', 'jumlah_kamar'], 'numerik tanpa kamar mandi'),
    (['luas_m2', 'lokasi_freq'], 'luas tanah + frekuensi lokasi'),
    (['luas_m2', 'luas_bangunan', 'jumlah_kamar', 'jumlah_kamar_mandi', 'lokasi_freq'],
     'numerik + frekuensi lokasi'),
]
hasil_reg = pd.DataFrame([evaluasi_regresi(k, l) for k, l in kandidat])
print(hasil_reg.sort_values('RMSE log').round(4).to_string(index=False))

                    Set fitur  Jumlah fitur  RMSE log     R2
   numerik + frekuensi lokasi             5    0.2995 0.5584
                semua numerik             4    0.3144 0.5134
    numerik tanpa kamar mandi             3    0.3351 0.4472
        luas tanah + bangunan             2    0.3461 0.4105
luas tanah + frekuensi lokasi             2    0.3784 0.2954
                 luas_m2 saja             1    0.3922 0.2430


## Langkah 5 — Perbandingan Encoding `lokasi`

Tiga representasi lokasi diuji dengan model yang sama (Ridge untuk regresi,
Logistic Regression untuk klasifikasi):

| Encoding | Cara kerja | Dimensi |
|---|---|---|
| Frequency | ganti lokasi dengan proporsi kemunculan | 1 |
| One-hot top-10 | 10 lokasi terbanyak + `Lainnya` | 11 |
| One-hot semua | setiap ejaan menjadi kolom | ratusan |

One-hot semua lokasi **tidak** dipakai di model final karena jumlah kolomnya jauh lebih
besar dari jumlah fitur lain dan banyak lokasi hanya muncul beberapa kali.

In [7]:
from sklearn.linear_model import LogisticRegression

top10 = df['lokasi_bersih'].value_counts().head(10).index
df['lokasi_top'] = np.where(df['lokasi_bersih'].isin(top10), df['lokasi_bersih'], 'Lainnya')

ohe_top = pd.get_dummies(df['lokasi_top'], prefix='lok', dtype=int)
ohe_all = pd.get_dummies(df['lokasi_bersih'], prefix='lok', dtype=int)

basis = df[['luas_m2', 'luas_bangunan', 'jumlah_kamar', 'jumlah_kamar_mandi']].reset_index(drop=True)

varian = {
    'tanpa lokasi': basis,
    'frequency': pd.concat([basis, df['lokasi_freq'].reset_index(drop=True)], axis=1),
    'one-hot top-10': pd.concat([basis, ohe_top.reset_index(drop=True)], axis=1),
    'one-hot semua': pd.concat([basis, ohe_all.reset_index(drop=True)], axis=1),
}

y_reg = df['log_harga'].reset_index(drop=True)
y_kls = df['kelas_harga'].reset_index(drop=True)

baris = []
for nama, X in varian.items():
    reg = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
    pred = cross_val_predict(reg, X, y_reg, cv=KFold(5, shuffle=True, random_state=RANDOM_STATE))
    rmse = np.sqrt(mean_squared_error(y_reg, pred))

    kls = make_pipeline(StandardScaler(with_mean=True), LogisticRegression(max_iter=1000))
    acc = cross_val_score(kls, X, y_kls, cv=KFold(3, shuffle=True, random_state=RANDOM_STATE),
                          scoring='accuracy').mean()
    baris.append({'Encoding lokasi': nama, 'Jumlah kolom': X.shape[1],
                  'RMSE log': rmse, 'Akurasi kelas': acc})

pd.DataFrame(baris).sort_values('RMSE log').round(4)

,Encoding lokasi,Jumlah kolom,RMSE log,Akurasi kelas
3,one-hot semua,592,0.2552,0.7408
1,frequency,5,0.2995,0.6827
2,one-hot top-10,15,0.3022,0.6869
0,tanpa lokasi,4,0.3144,0.6511


## Langkah 6 — Seleksi Fitur untuk Klasifikasi

Karena klasifikasi memakai kelas harga yang dipotong dari kuantil, hubungan fitur dengan
kelas biasanya lebih lemah daripada hubungan dengan harga kontinu. Perbandingan berikut
memakai Logistic Regression dengan 3-fold CV.

In [8]:
kandidat_kls = [
    (['luas_m2'], 'luas_m2 saja'),
    (['luas_m2', 'luas_bangunan'], 'luas tanah + bangunan'),
    (['luas_m2', 'luas_bangunan', 'jumlah_kamar', 'jumlah_kamar_mandi'], 'semua numerik'),
    (['luas_m2', 'luas_bangunan', 'jumlah_kamar', 'jumlah_kamar_mandi', 'lokasi_freq'],
     'numerik + frekuensi lokasi'),
]

hasil_kls = []
for kolom, label in kandidat_kls:
    X = df[kolom]
    model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
    skor = cross_val_score(model, X, df['kelas_harga'],
                           cv=KFold(3, shuffle=True, random_state=RANDOM_STATE),
                           scoring='accuracy').mean()
    hasil_kls.append({'Set fitur': label, 'Jumlah fitur': len(kolom), 'Akurasi CV': skor})

pd.DataFrame(hasil_kls).sort_values('Akurasi CV', ascending=False).round(4)

,Set fitur,Jumlah fitur,Akurasi CV
3,numerik + frekuensi lokasi,5,0.6827
2,semua numerik,4,0.6511
1,luas tanah + bangunan,2,0.6196
0,luas_m2 saja,1,0.5049


**Analisis JS03:**

- Korelasi terhadap log harga: **`luas_bangunan` paling kuat (r = 0,636)**, disusul `jumlah_kamar_mandi` (0,614), `luas_m2` (0,493), dan `jumlah_kamar` (0,490). Urutan ini juga muncul di MI: luas_bangunan 0,602; kamar mandi 0,368; kamar 0,332; luas tanah 0,254. Jadi justru **luas bangunan**, bukan luas tanah, yang paling informatif — berbeda dari dugaan awal bahwa `luas_m2` dominan.
- Semua VIF < 3 (tertinggi `jumlah_kamar_mandi` 2,61), sehingga **tidak ada multikolinearitas serius**; keempat fitur numerik boleh dipakai bersama.
- Regresi (Ridge, target log harga): set **numerik + frekuensi lokasi** terbaik (RMSE log 0,2995; R² 0,558). Menambah fitur dari 1 → 5 menurunkan RMSE dari 0,392 → 0,300.
- Encoding lokasi: **one-hot semua lokasi** (592 kolom) memberi RMSE log **0,2552** dan akurasi kelas **0,7408** — terbaik, tetapi jauh lebih besar. Frequency (1 kolom) memberi 0,2995/0,6827; one-hot top-10 0,3022/0,6869; tanpa lokasi 0,3144/0,6511. Artinya lokasi memang menambah informasi, terutama bila identitas kota dipertahankan penuh.
- Untuk klasifikasi, **numerik + frekuensi lokasi** memberi akurasi CV 0,6827; tanpa lokasi turun ke 0,6511.
- Trade-off: one-hot semua lokasi menambah ratusan kolom dan rawan overfitting pada lokasi jarang. Di JS15 dipakai `OneHotEncoder(min_frequency=30)` sebagai kompromi: identitas kota dipertahankan, kategori langka digabung.
- Kesimpulan fitur: **`luas_m2`, `luas_bangunan`, `jumlah_kamar`, `jumlah_kamar_mandi`, `lokasi`** (`lokasi_freq` untuk model linear; one-hot untuk pohon/JST), plus `kondisi` opsional.
